# 🤖 04 — Modelos Avançados: Random Forest & XGBoost

> **Credit Risk ML** · Notebook 4 de 5

**Objetivo:** superar o baseline com modelos de ensemble e selecionar o melhor.

| Etapa | Descrição |
|-------|-----------|
| 4.1 | Random Forest — treino e avaliação |
| 4.2 | XGBoost — treino e avaliação |
| 4.3 | Comparação de modelos |
| 4.4 | Feature importance comparativa |
| 4.5 | Seleção do melhor modelo |

---

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import joblib
from sklearn.ensemble import RandomForestClassifier
from xgboost          import XGBClassifier
from sklearn.metrics  import (
    roc_auc_score, roc_curve, auc, f1_score, precision_score, recall_score,
    accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report,
)
from pathlib import Path

plt.rcParams.update({
    'figure.facecolor':'#0F1117','axes.facecolor':'#161B22','axes.edgecolor':'#30363D',
    'axes.labelcolor':'#E6EDF3','xtick.color':'#8B949E','ytick.color':'#8B949E',
    'text.color':'#E6EDF3','grid.color':'#21262D','grid.linewidth':0.8,
    'figure.dpi':120,'font.family':'monospace',
})
C = {'lr':'#58A6FF','rf':'#3FB950','xgb':'#F78166',
     'accent':'#D2A8FF','warn':'#E3B341','muted':'#8B949E'}
SEED = 42
print('✅  Imports OK')

In [ ]:
# Carrega artefatos do notebook 02 e métricas do 03
splits     = joblib.load('../models/splits.pkl')
lr_metrics = joblib.load('../models/lr_metrics.pkl')
lr_model   = joblib.load('../models/logistic_regression.pkl')

X_train_sc = splits['X_train_sc']
X_test_sc  = splits['X_test_sc']
y_train    = splits['y_train']
y_test     = splits['y_test']
features   = splits['features']

# Calcula scale_pos_weight para XGBoost
n_neg = (y_train==0).sum()
n_pos = (y_train==1).sum()
scale_pos_weight = n_neg / n_pos

print(f'✅  Dados carregados | treino={len(X_train_sc):,} | teste={len(X_test_sc):,}')
print(f'   scale_pos_weight = {scale_pos_weight:.2f}')
print(f'   Baseline LR AUC = {lr_metrics["auc"]:.4f}')

In [ ]:
def evaluate_model(model, X_test, y_test, name, threshold=0.5):
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= threshold).astype(int)
    return {
        'name':name, 'model':model,
        'y_prob':y_prob, 'y_pred':y_pred,
        'auc'      : roc_auc_score(y_test, y_prob),
        'accuracy' : accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall'   : recall_score(y_test, y_pred, zero_division=0),
        'f1'       : f1_score(y_test, y_pred, zero_division=0),
    }

---
### 4.1 Random Forest

In [ ]:
def train_random_forest(X_train, y_train, save_path=None):
    """
    Random Forest com configurações para crédito:
      n_estimators=200     : suficiente para estabilidade sem custo excessivo
      max_depth=12         : controla overfitting
      min_samples_leaf=10  : exige pelo menos 10 amostras em cada folha
      class_weight=balanced: compensa desbalanceamento
      max_features='sqrt'  : decorrelaciona as árvores
    """
    model = RandomForestClassifier(
        n_estimators      = 200,
        max_depth         = 12,
        min_samples_split = 20,
        min_samples_leaf  = 10,
        max_features      = 'sqrt',
        class_weight      = 'balanced',
        random_state      = SEED,
        n_jobs            = -1,
    )
    t0 = time.time()
    model.fit(X_train, y_train)
    elapsed = time.time() - t0
    if save_path: joblib.dump(model, save_path)
    return model, elapsed

rf_model, rf_time = train_random_forest(
    X_train_sc, y_train, save_path='../models/random_forest.pkl'
)
rf_res = evaluate_model(rf_model, X_test_sc, y_test, 'Random Forest')

print(f'✅  Random Forest treinado em {rf_time:.1f}s')
print(f'   AUC      : {rf_res["auc"]:.4f}  (baseline: {lr_metrics["auc"]:.4f}  '
      f'Δ={rf_res["auc"]-lr_metrics["auc"]:+.4f})')
print(f'   F1       : {rf_res["f1"]:.4f}')
print(f'   Recall   : {rf_res["recall"]:.4f}')
print(f'\n{classification_report(y_test, rf_res["y_pred"], target_names=["Bom","Default"])}')

---
### 4.2 XGBoost

In [ ]:
def train_xgboost(X_train, y_train, X_val, y_val, scale_pos_weight, save_path=None):
    """
    XGBoost com configurações otimizadas:
      learning_rate=0.05   : pequeno → mais estável, mais árvores necessárias
      subsample=0.8        : 80% das amostras por árvore (reduz overfitting)
      colsample_bytree=0.8 : 80% das features por árvore (decorrelaciona)
      reg_alpha / reg_lambda: regularização L1 + L2
      early_stopping_rounds: para quando validação não melhora por 30 rounds

    NOTA: early_stopping_rounds vai no construtor (XGBoost >= 1.6)
    """
    model = XGBClassifier(
        n_estimators          = 400,
        max_depth             = 6,
        learning_rate         = 0.05,
        subsample             = 0.80,
        colsample_bytree      = 0.80,
        reg_alpha             = 0.1,
        reg_lambda            = 1.0,
        scale_pos_weight      = scale_pos_weight,
        eval_metric           = 'auc',
        early_stopping_rounds = 30,
        random_state          = SEED,
        n_jobs                = -1,
        verbosity             = 0,
    )
    t0 = time.time()
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    elapsed = time.time() - t0
    if save_path: joblib.dump(model, save_path)
    return model, elapsed

xgb_model, xgb_time = train_xgboost(
    X_train_sc, y_train, X_test_sc, y_test,
    scale_pos_weight, save_path='../models/xgboost.pkl'
)
xgb_res = evaluate_model(xgb_model, X_test_sc, y_test, 'XGBoost')

print(f'✅  XGBoost treinado em {xgb_time:.1f}s')
print(f'   Best iteration : {xgb_model.best_iteration}')
print(f'   AUC            : {xgb_res["auc"]:.4f}  '
      f'(baseline: {lr_metrics["auc"]:.4f}  Δ={xgb_res["auc"]-lr_metrics["auc"]:+.4f})')
print(f'   F1             : {xgb_res["f1"]:.4f}')
print(f'\n{classification_report(y_test, xgb_res["y_pred"], target_names=["Bom","Default"])}')

---
### 4.3 Comparação de Modelos

In [ ]:
lr_res = evaluate_model(lr_model, X_test_sc, y_test, 'Logistic Regression')
all_results = [lr_res, rf_res, xgb_res]

# Tabela comparativa
df_comparison = pd.DataFrame([
    {k: v for k, v in r.items() if k not in ('model','y_prob','y_pred')}
    for r in all_results
]).set_index('name').round(4)

print('╔══════════════════════════════════════════════════════════════╗')
print('║                 COMPARAÇÃO DE MODELOS                      ║')
print('╚══════════════════════════════════════════════════════════════╝')
print(df_comparison.to_string())

best_model_name = df_comparison['auc'].idxmax()
print(f'\n🏆  Melhor por AUC: {best_model_name} ({df_comparison.loc[best_model_name,"auc"]:.4f})')

In [ ]:
# Gráfico de comparação completo
fig = plt.figure(figsize=(17, 6))
fig.patch.set_facecolor('#0F1117')
gs  = gridspec.GridSpec(1, 3, figure=fig, wspace=0.35)

model_colors = {'Logistic Regression':C['lr'],'Random Forest':C['rf'],'XGBoost':C['xgb']}

# ROC
ax1 = fig.add_subplot(gs[0])
ax1.plot([0,1],[0,1],'k--',lw=1,alpha=0.5)
for res in all_results:
    fpr,tpr,_ = roc_curve(y_test, res['y_prob'])
    ax1.plot(fpr, tpr, lw=2.2, color=model_colors[res['name']],
             label=f'{res["name"]} ({res["auc"]:.3f})')
ax1.set_title('Curvas ROC', fontsize=13, fontweight='bold', color='#E6EDF3')
ax1.set_xlabel('FPR'); ax1.set_ylabel('TPR')
ax1.legend(fontsize=9, framealpha=0.3); ax1.grid(alpha=0.2)

# Métricas agrupadas
ax2 = fig.add_subplot(gs[1])
metrics_cols = ['auc','f1','precision','recall']
x = np.arange(len(metrics_cols)); w = 0.25
for i, res in enumerate(all_results):
    vals = [res[m] for m in metrics_cols]
    bars = ax2.bar(x+i*w, vals, w, label=res['name'],
                   color=model_colors[res['name']], alpha=0.9, edgecolor='#0F1117')
    for bar, v in zip(bars, vals):
        ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                 f'{v:.2f}', ha='center', va='bottom', fontsize=7.5)
ax2.set_xticks(x+w); ax2.set_xticklabels([m.upper() for m in metrics_cols], fontsize=9)
ax2.set_ylim(0, 1.12); ax2.set_title('Métricas Comparativas', fontsize=13, fontweight='bold', color='#E6EDF3')
ax2.legend(fontsize=9, framealpha=0.3); ax2.grid(axis='y', alpha=0.2)

# Matrizes de confusão (3 em 1)
ax3 = fig.add_subplot(gs[2])
ax3.axis('off')
table_data = [[r['name'], f'{r["auc"]:.4f}', f'{r["f1"]:.4f}',
               f'{r["recall"]:.4f}', f'{r["precision"]:.4f}']
              for r in all_results]
tbl = ax3.table(cellText=table_data,
                colLabels=['Modelo','AUC','F1','Recall','Precision'],
                cellLoc='center', loc='center')
tbl.auto_set_font_size(False); tbl.set_fontsize(10)
tbl.scale(1, 2.2)
for (row, col), cell in tbl.get_celld().items():
    cell.set_facecolor('#161B22' if row > 0 else '#21262D')
    cell.set_text_props(color='#E6EDF3')
    cell.set_edgecolor('#30363D')
ax3.set_title('Resumo', fontsize=13, fontweight='bold', color='#E6EDF3', pad=15)

plt.suptitle('Comparação Completa dos Modelos',
             fontsize=16, color=C['accent'], fontweight='bold')
plt.savefig('../reports/model_comparison.png', dpi=150, bbox_inches='tight', facecolor='#0F1117')
plt.show()

---
### 4.4 Feature Importance Comparativa

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.patch.set_facecolor('#0F1117')

for ax, (model, color, name) in zip(axes, [
    (rf_model,  C['rf'],  'Random Forest'),
    (xgb_model, C['xgb'], 'XGBoost'),
]):
    imp = pd.Series(model.feature_importances_, index=features).sort_values(ascending=True)
    colors_bar = [C['accent'] if v > imp.quantile(0.75) else color for v in imp.values]
    ax.barh(imp.index, imp.values, color=colors_bar, edgecolor='#0F1117', alpha=0.9)
    ax.set_title(f'Feature Importance — {name}', fontsize=12, fontweight='bold', color='#E6EDF3')
    ax.set_xlabel('Importância (Gini)', fontsize=11)
    ax.grid(axis='x', alpha=0.25)
    # Destaca features novas
    new_feats = ['loan_to_income','monthly_pay_ratio','risk_score',
                 'stability_index','high_risk_flag','inq_per_account']
    for label in ax.get_yticklabels():
        if label.get_text() in new_feats:
            label.set_color(C['accent'])
            label.set_fontweight('bold')

plt.suptitle('Feature Importance (roxo = features derivadas de engenharia)',
             fontsize=13, color=C['accent'], fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/feature_importance.png', dpi=150, bbox_inches='tight', facecolor='#0F1117')
plt.show()

---
### 4.5 Seleção do Melhor Modelo

In [ ]:
# Seleciona o melhor por AUC (critério principal)
best_res  = max(all_results, key=lambda r: r['auc'])
best_model = best_res['model']

joblib.dump(best_model, '../models/best_model.pkl')
# Salva todos os resultados para o notebook 05
joblib.dump({
    'all_results'     : all_results,
    'best_model_name' : best_res['name'],
    'best_model_auc'  : best_res['auc'],
    'scale_pos_weight': scale_pos_weight,
}, '../models/model_results.pkl')

print('╔══════════════════════════════════════════════════════════╗')
print('║         SELEÇÃO DO MELHOR MODELO                        ║')
print('╠══════════════════════════════════════════════════════════╣')
print(f'║  Modelo selecionado : {best_res["name"]:<32} ║')
print(f'║  ROC-AUC            : {best_res["auc"]:.4f}                          ║')
print(f'║  F1-Score           : {best_res["f1"]:.4f}                          ║')
print(f'║  Recall             : {best_res["recall"]:.4f}                          ║')
print('║                                                          ║')
print('║  Arquivos salvos:                                        ║')
print('║    models/best_model.pkl                                ║')
print('║    models/model_results.pkl                             ║')
print('║    reports/model_comparison.png                         ║')
print('║    reports/feature_importance.png                       ║')
print('║                                                          ║')
print('║  → Próximo: 05_decision_simulation.ipynb               ║')
print('╚══════════════════════════════════════════════════════════╝')